In [46]:
using LinearAlgebra
using DifferentialEquations
using GLMakie
using StaticArrays

using VMRobotControl

# Building a simple robot with 2 prismatic joints

In [50]:
# Kinematic structure definition

robot = Mechanism{Float64}("PrismaticRobot")
F0 = root_frame(robot)
F1 = add_frame!(robot; id="L1_frame")
F2 = add_frame!(robot; id="L2_frame")

J = Prismatic(SVector(1., 0., 0.))

add_joint!(robot, J; parent=F0, child=F1, id="J1")
add_joint!(robot, J; parent=F0, child=F2, id="J2")

# r = compile(robot)
# kcache = Observable(new_kinematics_cache(r))

# fig = Figure(size=(800, 600))
# ls = LScene(fig[1, 1]; show_axis=true)  # 3D interactive scene
# cam3d!(ls)  
# robotsketch!(ls, kcache; scale = 0.5)

# t = 0.0
# q = Float64[-0.2, 0.2]
# kinematics!(kcache[], t, q)
# notify(kcache)

# display(fig)

"J2"

In [51]:
# mass and inertia

add_coordinate!(robot, FrameOrigin(F1); id="f1_centre_of_mass")
add_coordinate!(robot, FrameOrigin(F2); id="f2_centre_of_mass")

add_component!(robot, PointMass(1.0, "f1_centre_of_mass"); id="f1_mass")
add_component!(robot, PointMass(1.0, "f2_centre_of_mass"); id="f2_mass")

# I_mat = @SMatrix [
#     0.1  0.    0.  ;
#     0.    0.1  0.  ;
#     0.    0.    0.1
# ]

# add_inertia!(robot, F1, I_mat; id="L1_inertia")
# add_inertia!(robot, F2, I_mat; id="L2_inertia")

# for i = 1:2
#     add_coordinate!(robot, JointSubspace("J$i"); id="J$i")
#     add_component!(robot, LinearDamper(0.5, "J$i"); id="J$(i)_damper")
# end

"f2_mass"

# Linking both frames with a simple spring

In [52]:
vms1 = VirtualMechanismSystem("myVMS1", robot)

add_coordinate!(vms1, CoordDifference(".robot.f1_centre_of_mass", ".robot.f2_centre_of_mass"); id="bidirectional position error");

K = SMatrix{3, 3}(1., 0., 0., 0., 1., 0., 0., 0., 1.)
add_component!(vms1, LinearSpring(K, "bidirectional position error"); id="bidirectional spring");
D = SMatrix{3, 3}(5., 0., 0., 0., 5.0, 0., 0., 0., 5.)
add_component!(vms1, LinearDamper(D, "bidirectional position error"); id="bidirectional damper")

"bidirectional damper"

In [57]:
# SIMULATION
tspan = (0., 15.)
vms_compiled = compile(vms1)
q = ([-1.0, 1.0], zero_q(vms_compiled.virtual_mechanism)) 
q̇ = (zero_q̇(vms_compiled.robot), zero_q̇(vms_compiled.virtual_mechanism)) 
g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan)
sol = solve(prob, Tsit5(), progress=true; maxiters=1e6, abstol=1e-6, reltol=1e-6);

# ANIMATION
fig = Figure(size=(700, 750))
ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
cam = cam3d!(ls, camera=:perspective, center=false)  
cam.lookat[] = [-0.06, 0.07, 0.06]
cam.eyeposition[] = [-0.75, 1.0, 0.6]

plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; fps = 25);

# Linking both frames with unidirectional spring

In [58]:
vms2 = VirtualMechanismSystem("myVMS2", robot)

add_coordinate!(vms2, ShadowCoord(".robot.f1_centre_of_mass"); id="f1_shadow_coord");
add_coordinate!(vms2, CoordDifference("f1_shadow_coord", ".robot.f2_centre_of_mass"); id="unidirectional position error");

K = SMatrix{3, 3}(1., 0., 0., 0., 1., 0., 0., 0., 1.)
add_component!(vms2, LinearSpring(K, "unidirectional position error"); id="Linear Spring");
D = SMatrix{3, 3}(5., 0., 0., 0., 5.0, 0., 0., 0., 5.)
add_component!(vms2, LinearDamper(D, "unidirectional position error"); id="Linear Damper")

"Linear Damper"

In [60]:
# SIMULATION
#tspan = (0., 15.)
vms_compiled = compile(vms2)
#q = ([-1.0, 1.0], zero_q(vms_compiled.virtual_mechanism)) 
#q̇ = (zero_q̇(vms_compiled.robot), zero_q̇(vms_compiled.virtual_mechanism)) 
#g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan)
sol = solve(prob, Tsit5(), progress=true; maxiters=1e6, abstol=1e-6, reltol=1e-6);

# ANIMATION
# fig = Figure(size=(700, 750))
# ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
# cam = cam3d!(ls, camera=:perspective, center=false)  
# cam.lookat[] = [0.2, 0.2, 0.2]
# cam.eyeposition[] = [-1.25, 0.88, 0.7]

plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; fps = 25);

ErrorException: Screen not open!